# 43008: Reinforcement Learning

## WOW Naval Simulation: Multi-Agent PPO (MAPPO) with a curriculum

This notebook trains a fleet of warships to win naval battles in our Unity game. Every ship is an agent, and all of them learn together with **MAPPO**, the multi-agent version of the PPO method from Week 10. Training follows a **curriculum**: it starts with a 1v1 battleship duel and moves up, stage by stage, to random fleet battles on random maps.

The objective is the game's own: **get to the centre of the capture circle and hold it to score points, sink enemy ships, and do not lose your own.**

### What you will learn?
* How a Unity game becomes a reinforcement learning environment (observation, actions, reward)
* Define hyperparameters for MAPPO
* Train a team of agents with a centralised critic, self-play and a curriculum
* Performance evaluation: learning curve, win rate per stage, battle replays

---

### Multi-Agent PPO (MAPPO)

Reference: https://arxiv.org/abs/2103.01955 (Yu et al., *The Surprising Effectiveness of PPO in Cooperative Multi-Agent Games*, NeurIPS 2022)

Original PPO article: [Link]("https://arxiv.org/pdf/1707.06347v2.pdf")

MAPPO is the PPO you used on BipedalWalker and Atari, with three changes for a team:

| | PPO (Week 10) | MAPPO (this notebook) |
|---|---|---|
| Agents | one | every ship of the fleet (1 to 8) |
| Actor (policy) | one network | **one network shared by every ship** - a battleship and a destroyer act differently because they *observe* differently |
| Critic (value) | sees the same observation as the actor | **centralised**: sees the true state of the whole battle (both fleets, every circle), only during training |
| Reward | the agent's own | each ship's own terms + the team's shared terms (captures, kills, the win) |
| An agent dies | the episode ends | **death masking**: the sunk ship stops training the actor, but the critic keeps learning the team's value until the battle ends |

Because the actor only uses what its own ship can see (fog of war included), the trained policy can play the real game - this is called *centralised training, decentralised execution*.

The training loop is exactly PPO's:

1. **Collect** `n_steps` decisions from every battle with the current policy.
2. **Advantages** with GAE: how much better each action turned out than the critic expected.
3. **Update** for `n_epochs` x `n_minibatches`: the clipped policy loss, the critic's value loss, and an entropy bonus.

The paper's five implementation recommendations are all here: **value normalisation** (`ValueNorm` - the critic learns returns rescaled to about unit size), an **agent-specific global state** for the critic, **few minibatches and several epochs** (4 x 10), **PPO clipping** at 0.2, and **death masking**. Setting `centralised_critic=False` turns it into **IPPO** (the critic sees only the ship's own view) - the baseline MAPPO is compared against.

All of it is in `simple_mappo/mappo.py` (about 430 lines) - read it alongside this notebook.

## The environment

```
 Python (this notebook)                          Unity headless player (one battle per process)
 ┌───────────────────────┐   actions, every ship   ┌──────────────────────────────────────────┐
 │ MAPPO actor + critic  │ ──────────────────────▶ │ simulates 1 second of game time          │
 │                       │ ◀────────────────────── │ (ships, gunnery, torpedoes, fog of war)   │
 └───────────────────────┘ observation, reward     └──────────────────────────────────────────┘
                           numbers, battle over?          × n_envs battles in parallel
```

`NavalEnv` (in `simple_mappo/env.py`) runs `n_envs` battles side by side - the same idea as `make_atari_env(n_envs=4)` in the Atari notebooks. It launches the game's **headless player** (no window, no graphics) and talks to it over TCP. The enemy fleet is flown by the game's own rule-based AI - or, in self-play battles, by a frozen copy of our own network.

A fleet wins by reaching 1000 points (holding a circle earns points every second, each kill 12) or by sinking the whole enemy fleet. **There are no draws**: if the points are level when the clock runs out, the battle goes to overtime and the first side to pull ahead wins.

**Speed.** Training does not run at x1. Each simulated frame is a fixed `sim_dt` of game time and frames run as fast as the CPU allows. With `sim_dt = 0.08` (the physics step the game itself uses at x8 time compression) one battle runs at about **260x real time** - a 6-minute battle takes about 1.4 s - with the same battle results as the normal 0.02 step.

**Observation** - what each ship knows (`obs["local"]`, 758 numbers):

| Part | What it holds |
|---|---|
| self | class, hit points, 7 damaged systems, speed, heading, position, reload timers, ammunition, every consumable (ready / active / charges), fires, floods, the match state (time left, both scores, weather, mode) |
| allies | position, distance, class, hit points, what they are doing - for each friendly ship |
| contacts | position, distance, class, aspect, in gun / torpedo range, line of sight - for each enemy ship your team has spotted |
| capture circles | position, **distance**, radius, owner, capture progress, **"I am inside"** |
| obstacles | the nearest islands, rocks and smoke clouds |

**One method for every stage**: the observation always has room for 8 ships a side and 5 circles, so the *same* network - one model, one checkpoint - trains through every stage of the curriculum (and any stage added later); a duel just leaves seven ship slots empty, with a mask saying so. The critic gets `obs["state"]` (2668 numbers): the *true* state of both fleets and every circle, plus that ship's own view and which ship it is.

**Actions** - every ship picks one option from each of 6 heads (illegal options are masked out):

| Head | Options |
|---|---|
| move | keep course, 8 compass directions, close in, broadside, open range, regroup, take cover, return to port, **sail to circle k** |
| speed | full, 2/3, 1/3, stop, astern |
| target | auto, or focus enemy contact k |
| fire | fire at will, hold fire |
| torpedo | none, launch at the target |
| ability | none, one of 12 consumables (shell type, smoke, radar, repair...), dive, surface |

### The reward

Defined in `simple_mappo/rewards.py` - the one file to edit when you want the fleet to behave differently. **Ship** terms are earned by one ship for what it did; **team** terms are shared by the whole fleet.

| Term | Weight | Kind | Paid for |
|---|---|---|---|
| `in_circle` | +0.05 | ship | every second inside a capture circle |
| `approach_circle` | +1 | ship | every km sailed towards the nearest circle's centre (minus when sailing away) |
| `zone_captured` / `zone_lost` | +1 / -1 | team | a circle captured / lost |
| `enemy_sunk` | +1 | team | each enemy ship sunk |
| `ship_lost` | -1 | team | each of our ships sunk |
| `damage_dealt` | +0.5 | ship | per whole enemy hull of damage this ship did |
| `damage_taken` | -0.25 | ship | per whole hull of damage this ship took |
| `friendly_fire` | -1 | ship | damage done to our own side |
| `win` / `loss` | +5 / -5 | team | the result, on the last step |

The circle terms are the largest steady signal on purpose: with smaller circle weights (0.02 and 0.2) and kills worth ±2, the fleet fought but never found the circle in 250 battles; with these it was sailing into the circle within about 150. (Stage 0, the duel, switches the circle terms off - see the curriculum below.)

### The curriculum

Defined in `simple_mappo/curriculum.py`. Training moves to the next stage once the fleet's **rolling win rate against the rule-based AI** reaches the stage's requirement (self-play battles do not count).

| Stage | Training scenario | Fleet | Environment | Opponent | Promotion / stopping requirement |
|---|---|---|---|---|---|
| 0 - Battleship Duel | `bb_duel` | 1v1 battleships | Open-sea duel scenario | Recruit rule-based AI | ≥55% rolling win rate after at least 100 qualifying episodes |
| 1 - King of the Hill | `koth_3v3` | 3v3 mixed fleet | Single-objective combat scenario | Veteran rule-based AI | ≥65% rolling win rate after at least 100 qualifying episodes |
| 2 - Archipelago | `archipelago_3v3` | 3v3 mixed fleet | Islands / archipelago scenario | Veteran rule-based AI | ≥65% rolling win rate after at least 150 qualifying episodes |
| 3 - Full Domination | `domination_6v6` | 6v6 fleet | Procedural battlefield with random weather | Elite rule-based AI + self-play | ≥60% rolling win rate after at least 200 qualifying Elite rule-AI episodes |
| 4 - Open / Generalisation | `open` | Random 4-8 ships per side | Random map preset, terrain density and weather | Elite rule-based AI + self-play | **Final stage**: stop at ≥60% rolling win rate after at least 250 qualifying held-out evaluation episodes against the Elite rule-based AI |

* **One model for every stage**: promotion only changes the battles; the same network keeps training. The one stage setting that changes the reward: **stage 0 does not pay for the circle**, because in the duel the circle lies between the two battleships and racing to it sends the ship bow-on into the enemy's guns (sailing to the circle wins 33% of duels, broadside and fire at will 70%).
* **Self-play** (stages 3-4): half the battles are against a frozen copy of our own network, refreshed every 10 updates, so the fleet also learns to beat a smarter, changing opponent.
* **Held-out evaluation** (stage 4): every 100,000 timesteps the fleet plays 50 battles against the Elite AI on generated maps whose seeds training never uses; training stops once the last 250 of those reach 60%.
* **More stages**: add dicts to the list - there is no limit. The last stage keeps training until its stopping rule is met (drop `stop_win_rate` to train on it forever).

---
## Setup

### Running on Amazon SageMaker

1. On your computer, in the repository: build the headless player once (`Training/build_player.sh`), then run `Training/package_sagemaker.sh`. It writes `wow_sagemaker.zip` (code + scenarios + the player, about 40 MB).
2. Upload `wow_sagemaker.zip` to SageMaker Studio (drag it into the file browser) and run the cell below.
3. Pick an instance with **many CPUs** for the JupyterLab space (e.g. `ml.c5.4xlarge`, 16 vCPU): every parallel battle is one Unity process on its own core. No GPU is needed; the networks are small.
4. The whole curriculum takes millions of timesteps - for that, use the **training job** at the end of this notebook, so it does not depend on the notebook staying open.

On your own computer, skip this cell and open the notebook from the repository's `Training/` folder.

In [ ]:
import os
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
    if not os.path.isdir(os.path.join(home, "wow")):
        import zipfile
        assert os.path.exists(os.path.join(home, "wow_sagemaker.zip")), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(os.path.join(home, "wow_sagemaker.zip")).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training
print(os.getcwd(), "- glibc", os.confstr("CS_GNU_LIBC_VERSION"))   # the Unity player needs glibc 2.35+

## Install Packages and Dependencies

In [ ]:
# PyTorch, NumPy, pandas, matplotlib and Pillow are already on SageMaker images and Anaconda; install only
# what is missing (PyTorch as the CPU build - training runs on the CPU and the GPU build is ~2.5 GB)
import importlib.util
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

## Importing required libraries and modules

In [ ]:
import os
import json
import copy
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from simple_mappo import (NavalEnv, MAPPO, Monitor, BaseCallback, CurriculumCallback, DEFAULT_CURRICULUM,
                          REWARD_WEIGHTS, TeamStorage, TeamSyncCallback, evaluate, plot_results, plot_curriculum,
                          plot_reward_terms, plot_progress, record_battle, show_videos, view, find_unity_binary)

print(f"{torch.__version__=}  cuda={torch.cuda.is_available()}  CPUs={os.cpu_count()}")
print("headless player:", find_unity_binary())

## Team checkpoints

The whole team trains **one shared line of training** in turns. Its checkpoints and logs live in one S3 folder,
`<TEAM_STORAGE>/runs/<RUN_NAME>/`. When you start, the notebook pulls the latest checkpoint from there and continues it (curriculum stage included); while you train it pushes a new checkpoint every `save_interval` timesteps, and once more at the end. The learning curve in `logs/monitor.csv` keeps growing across everybody's sessions.

* Everyone must use the **same `TEAM_STORAGE` and `RUN_NAME`** to continue the same training. On SageMaker the default is your account's SageMaker bucket - if teammates log in to *different* AWS accounts, create one bucket, give everyone access, and put its name here.
* Only one person can train a run at a time: if a teammate is training it, the notebook says so and stops. Use another `RUN_NAME` to train something different in parallel (e.g. a reward experiment).
* `TEAM_STORAGE = None` trains on this machine only.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates in the shared run
RUN_NAME = "curriculum"              # the line of training the team takes turns extending
if ON_SAGEMAKER:
    import sagemaker
    TEAM_STORAGE = f"s3://{sagemaker.Session().default_bucket()}/wow-mappo"    # or "s3://<your-team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<your-team-bucket>/wow-mappo" (needs boto3), or a shared folder

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
if storage:
    print("team run:", storage.location)
    print("last push:", storage.info() or "none yet - you will start it")

## Create Directories for Model and Logs Storage

In [ ]:
# Change the paths based on your directory structure
run_dir = f"runs_simple/{RUN_NAME}"
log_dir = os.path.join(run_dir, "logs/")
models_dir = os.path.join(run_dir, "models/")
video_dir = os.path.join(run_dir, "videos/")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(video_dir, exist_ok=True)

## Callback for Model Checkpointing

The same callback as Weeks 9 and 10. One difference: with `n_envs` parallel battles `num_timesteps` grows by `n_envs` per step, so it seldom lands exactly on a multiple of `save_interval` - this version saves whenever it *crosses* one.

In [ ]:
class SaveOnIntervalCallback(BaseCallback):
    def __init__(self, save_interval: int, save_path: str, verbose=1):
        super().__init__(verbose)
        self.save_interval = save_interval
        self.save_path = save_path
        self.saved = 0

    def _on_training_start(self) -> None:
        self.saved = self.num_timesteps // self.save_interval
        if self.num_timesteps == 0:
            self.model.save(os.path.join(self.save_path, "model_0"))   # the untrained policy, for the "beginning" replay

    def _on_step(self) -> bool:
        if self.num_timesteps // self.save_interval > self.saved:
            self.saved = self.num_timesteps // self.save_interval
            save_file = os.path.join(self.save_path, f'model_{self.num_timesteps}')
            self.model.save(save_file)
            if self.verbose > 0:
                print(f'Saving model to {save_file}.pt')
        return True

## Configuration and Environment Initialization

A **timestep** is one decision of one battle (every ship decides once; one decision = `decision_period` seconds of game time). With `n_envs = 8`, 8 timesteps are collected per step. 300,000 timesteps take about 30-40 minutes on 8 CPU cores - enough to see stage 0 learning; the whole curriculum needs millions (use the training job).

In [ ]:
# Define hyper parameters for MAPPO
config = OrderedDict([
    # the battles
    ('curriculum', copy.deepcopy(DEFAULT_CURRICULUM)),   # the stages above (edit, or add your own)
    ('start_stage', 0),
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles in parallel, one Unity process each
    ('decision_period', 1.0),          # game seconds between decisions
    ('sim_dt', 0.08),                  # game seconds per frame: 0.02 normal, 0.08 fastest stable
    # MAPPO
    ('n_steps', 256),                  # decisions per battle before each update
    ('n_minibatches', 4),              # each update splits the collected data into 4 minibatches...
    ('n_epochs', 10),                  # ...10 times: 40 gradient steps, the same for a 1v1 duel or an 8v8 battle
    ('gamma', 0.99),
    ('gae_lambda', 0.95),
    ('clip_range', 0.2),
    ('ent_coef', 0.01),
    ('vf_coef', 0.5),
    ('learning_rate', 3e-4),
    ('hidden_size', 256),
    ('centralised_critic', True),      # True = MAPPO; False = IPPO (critic sees only the ship's own view), to compare
    ('opponent_refresh', 10),          # self-play: updates between refreshing the frozen opponent
    ('n_timesteps', 300_000),          # Adjust this depending on your training limit
])
pd.DataFrame(config['curriculum']).fillna("")

In [ ]:
# The reward weights (see the table above). Change one here, e.g. reward_weights['in_circle'] = 0.1
reward_weights = OrderedDict(REWARD_WEIGHTS)
pd.Series(reward_weights, name="weight").to_frame()

In [ ]:
def make_env():
    # launches n_envs headless Unity players; the observation is sized for the biggest stage
    env = NavalEnv(stages=config['curriculum'], start_stage=config['start_stage'], n_envs=config['n_envs'],
                   decision_period=config['decision_period'], sim_dt=config['sim_dt'],
                   reward_weights=reward_weights, unity_log_dir=os.path.join(run_dir, "unity_logs"))
    return Monitor(env, log_dir)

env = make_env()
print("battles in parallel:", env.n_envs, " ship slots per side:", env.n_agents)
print("actor input (local obs):", env.local_dim, " critic input (state):", env.state_dim)
print("action heads:", dict(zip(env.head_names, env.head_sizes)))
print("current stage:", env.stages[env.stage]["name"])

### Look inside one observation

Every number has a name - Unity sends the feature names with the environment's description. Here are the first ship's capture-circle features, which the `in_circle` and `approach_circle` rewards are computed from.

In [ ]:
obs = env.reset()
print({k: v.shape for k, v in obs.items()})

raw = env.raw_arrays(0)                         # everything Unity sent for battle 0, both fleets
zone_names = env.spec.features["zone"]
circle = raw["zones"][0, 0, 0]                  # [fleet 0, ship 0, circle 0]
pd.Series(circle, index=zone_names, name="ship 0, circle 0").to_frame().T

## Baselines

Before training, measure what a policy has to beat on the first stage:
* **Random**: a random legal option on every head.
* **Scripted**: every ship sails to the circle at full speed and fires at will (a sensible hand-written tactic).

In [ ]:
class GoToCircle:
    # move head: 15 fixed options, then one per circle; speed 0 = full; target 0 = auto; fire 0 = at will
    def predict(self, obs, deterministic=False):
        E, N = obs["alive"].shape
        actions = np.zeros((E, N, len(env.head_sizes)), dtype=np.int32)
        actions[..., 0] = np.where(obs["action_mask"][..., 15] > 0.5, 15, 0)
        return actions

baselines = {"random": evaluate(None, env, n_battles=24),
             "scripted: go to circle": evaluate(GoToCircle(), env, n_battles=24)}
pd.DataFrame(baselines).T[["stage", "win_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost"]]

(The baseline battles are logged as evaluation battles; the learning-curve plots leave them out.)

## Model Initialization and Training

In [ ]:
# Continue the team's training if the shared run already has a checkpoint, otherwise start a new model
latest = storage.pull(run_dir) if storage else None
if latest:
    model = MAPPO.load(latest, env=env, log_dir=log_dir, verbose=1)
    print(f"continuing at {model.num_timesteps:,} timesteps on stage '{env.stages[env.stage]['name']}'")
else:
    model = None
if storage:
    storage.lock()                   # stops here if a teammate is training this run right now

In [ ]:
# Create the MAPPO model (skipped when continuing the team's run)
if latest is None:
    model = MAPPO(
        env,
        learning_rate=config['learning_rate'],
        n_steps=config['n_steps'],
        n_minibatches=config['n_minibatches'],
        n_epochs=config['n_epochs'],
        gamma=config['gamma'],
        gae_lambda=config['gae_lambda'],
        clip_range=config['clip_range'],
        ent_coef=config['ent_coef'],
        vf_coef=config['vf_coef'],
        hidden_size=config['hidden_size'],
        centralised_critic=config['centralised_critic'],
        opponent_refresh=config['opponent_refresh'],
        verbose=1,
        log_dir=log_dir,
    )

save_interval = config['n_timesteps'] // 3
callback = [SaveOnIntervalCallback(save_interval, save_path=models_dir),
            CurriculumCallback()]          # promotes to the next stage, and applies stage 4's stopping rule
if storage:
    callback.append(TeamSyncCallback(storage, run_dir, push_every=save_interval))   # share progress with the team

print(model.actor)
print(model.critic)

`n_timesteps` is how many timesteps to train *in this session* (continuing a team run adds to what is already there).

Each printed table is one update. What to watch:

| Line | Meaning | Healthy |
|---|---|---|
| `stage` | the curriculum stage new battles start on | goes up |
| `ep_rew_mean` | average reward per ship per battle | goes up |
| `win_rate` | share of battles won against the rule AI since the last table | goes up |
| `win_rate_self_play` | the same against the frozen copy (stages 3-4) | about 0.5 |
| `entropy` | how random the policy still is (6 heads summed) | falls slowly, not to 0 |
| `approx_kl` | how far one update moved the policy | about 0.005 - 0.03 |
| `clip_fraction` | share of samples PPO's clip stopped | below about 0.3 |
| `explained_variance` | how well the critic predicts the returns | rises towards 1 |

In [ ]:
# Start the training (a progress table is printed every log_interval updates)
model.learn(total_timesteps=model.num_timesteps + config['n_timesteps'], callback=callback, log_interval=5)

## Save Model

Saves the final model and hands the run back to the team. **If you stop training early (Kernel > Interrupt), run this cell too**, so your progress is pushed and the lock released.

In [ ]:
# Save the final model (it remembers the curriculum stage it reached)
model.save(os.path.join(models_dir, "model_final"))
model.save(os.path.join(models_dir, "model_latest"))
if storage:
    storage.push(run_dir, model)     # your checkpoint and logs, for the team
    storage.release()                # a teammate can continue from here

## Plot the Learning Curve

In [ ]:
plot_results(log_dir)

Win rate against the rule AI, one colour per curriculum stage - a new colour is a promotion:

In [ ]:
plot_curriculum(log_dir)

Which reward terms the fleet is earning, battle by battle. A rising `in_circle` means the ships learned to hold the circle; `ship_lost` moving towards 0 means they stopped dying.

In [ ]:
plot_reward_terms(log_dir)

In [ ]:
plot_progress(log_dir)

## Performance Evaluation

Fresh battles against the rule AI on every stage reached so far, with sampled actions (a young policy is often close to 50/50 on choices like fire / hold fire, and always taking the likelier option can mean never firing).

In [ ]:
results = {f"{name} ({r['stage']})": r for name, r in baselines.items()}
for s in range(env.stage + 1):
    results[f"MAPPO on {env.stages[s]['name']}"] = evaluate(model, env, n_battles=24, stage=s)
pd.DataFrame(results).T[["battles", "win_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost"]]

## Record and Display Replays

Top-down replays of one battle at the beginning, middle and end of training, each on the stage that model had reached. Cyan is our fleet, red the enemy, the dashed ring a capture circle (its colour shows the owner), and the green bar each ship's hit points.

In [ ]:
view(models_dir, env, video_folder=video_dir)

In [ ]:
# Display the replays created at different stages of the trained model
for stage in ["beginning", "middle", "end"]:
    show_videos(video_dir, prefix=f'mappo-naval-{stage}')

In [ ]:
env.close()

---
## Next steps

* **Keep training**: just run the notebook again - with team checkpoints it pulls the newest one (yours or a teammate's) and continues. Without them: `model = MAPPO.load(os.path.join(models_dir, "model_final"), env=env)` then `model.learn(...)`.
* **Add a stage** (there is no limit): append a dict to `config['curriculum']`, e.g. a Capture and Control stage:
  `{"name": "5 Capture and Control", "procedural": {"mode": 3, "preset": "random", "ships": [6, 8]}, "difficulty": 2, "self_play": 0.5, "promote_win_rate": 0.6, "window": 200}`.
  Move the `stop_win_rate` rule to whichever stage is last. A model is tied to the observation size, so adding a stage with *more* ships or circles than any existing one needs a new model.
* **Change the behaviour**: edit the weights in `simple_mappo/rewards.py` (or `reward_weights` above) and compare the learning curves.

### Watch it live in the Unity editor (on your own computer)

Open the project, select the `GameBootstrap` object, tick **rlTrainingServer**, press Play, then:

```python
live = NavalEnv(stages=config['curriculum'], start_stage=1, ports=[5005])   # connects to the editor
model = MAPPO.load(os.path.join(models_dir, "model_final"), env=live)
obs = live.reset()
for _ in range(400):
    obs, rewards, dones, infos = live.step(model.predict(obs))
```

The editor plays at the training speed; add `sim_dt=0.02` to `NavalEnv(...)` to watch at normal speed.

---
## (Optional) Train the whole curriculum as a SageMaker Training Job

Runs `train_simple.py` on a separate machine, so training does not depend on this notebook staying open. The headless player goes up as the `unity` input channel. The job uses the **same team checkpoints** as the notebook: it pulls the run's latest checkpoint, pushes every `save-interval` timesteps and at the end - so a teammate can continue a job's training in a notebook and the other way round. (Don't run a job and a notebook on the same `RUN_NAME` at once: the second one is refused.) Run this from SageMaker Studio inside `~/wow/Training`.

In [ ]:
import shutil
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()
assert TEAM_STORAGE, "set TEAM_STORAGE in the Team checkpoints cell - it is where the job saves its checkpoints"

# the headless player, as the job's "unity" input channel
unity_s3 = session.upload_data("../Builds/NavalTrainer", bucket=session.default_bucket(), key_prefix="wow-mappo/unity")

# only the code the job needs (not runs_simple/, which holds your local checkpoints)
job_source = os.path.expanduser("~/wow_job_source")
shutil.rmtree(job_source, ignore_errors=True)
os.makedirs(job_source)
for item in ["simple_mappo", "naval_rl", "scenarios", "train_simple.py", "requirements.txt"]:
    if os.path.isdir(item):
        shutil.copytree(item, os.path.join(job_source, item), ignore=shutil.ignore_patterns("__pycache__"))
    else:
        shutil.copy(item, job_source)

estimator = PyTorch(
    entry_point="train_simple.py",
    source_dir=job_source,
    role=role,
    framework_version="2.5",                          # 2.4+ images are Ubuntu 22.04: the Unity player needs its glibc 2.35
    py_version="py311",
    instance_type="ml.c5.4xlarge",                    # 16 vCPU, 32 GB: 8 battles in parallel
    instance_count=1,
    hyperparameters={"total-timesteps": 20_000_000, "n-envs": 8, "curriculum": "default",
                     "team-storage": TEAM_STORAGE, "run-name": RUN_NAME, "user": f"{YOUR_NAME}-job",
                     "save-interval": 100_000},
    max_run=5 * 24 * 3600,                            # SageMaker's limit; the job resumes the team run when restarted
)
estimator.fit({"unity": unity_s3}, wait=False)
print("follow it in", storage.location, "- storage.info() shows the job's last push")